# BinaryMatchboxNet KWS — Colab bootstrap

Run the cells top to bottom. Cell 1 always hard-resets the working copy to
`origin/main`, so **anything edited inside Colab is discarded** — edit locally,
commit, push, then re-run cell 1.

Prereq: a GitHub PAT stored as a Colab secret named `GH_TOKEN`
(🔑 icon in the left sidebar → *Add new secret* → toggle *Notebook access*).
See `docs/colab_setup.md`.

> **After a `git pull` that changes `.py` files**, cell 1's `%autoreload`
> refreshes already-imported modules. If something still looks stale (rare),
> **Runtime → Restart session** and run from the top — a restart always wins.

In [ ]:
# --- 1. clone / sync the repo -------------------------------------------
# autoreload picks up code changes after a git pull WITHOUT restarting the
# runtime -- otherwise re-running an import cell reuses the stale cached
# module and your `git reset --hard` looks like it did nothing.
%load_ext autoreload
%autoreload 2

import os
from google.colab import userdata

GH_TOKEN = userdata.get('GH_TOKEN')
USER   = 'minochichic'
REPO   = 'KWS-AFE-Digital'
BRANCH = 'main'
DIR    = f'/content/{REPO}'

if not os.path.exists(DIR):
    !git clone -b {BRANCH} https://{GH_TOKEN}@github.com/{USER}/{REPO}.git {DIR}

%cd {DIR}
!git fetch origin
!git reset --hard origin/{BRANCH}
!git log -1 --oneline

In [ ]:
# --- 2. dependencies -----------------------------------------------------
# requirements-colab.txt deliberately omits torch/torchaudio: Colab's builds
# are matched to its CUDA runtime and reinstalling them breaks GPU support.
!pip install -q -r requirements-colab.txt

In [ ]:
# --- 3. environment check ------------------------------------------------
import torch, torchaudio, sys
print('python     ', sys.version.split()[0])
print('torch      ', torch.__version__)
print('torchaudio ', torchaudio.__version__)
print('cuda       ', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU only)')

if not torch.cuda.is_available():
    print('\n[!] No GPU. Runtime -> Change runtime type -> T4/A100 GPU.')

In [ ]:
# --- 4. unit tests (binary ops, AFE, model, config guardrails) -----------
!python -m pytest -q

In [ ]:
# --- 5. what will actually be built --------------------------------------
!python experiments/inspect_model.py configs/base.yaml

In [ ]:
# --- 6. persist checkpoints to Drive (optional) --------------------------
# Colab sessions are wiped on disconnect. Symlink runs/ into Drive so a long
# sweep survives a dropped runtime. Checkpoints are gitignored on purpose --
# do not push them.
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/kws_runs
!rm -rf runs && ln -s /content/drive/MyDrive/kws_runs runs
!ls -la runs/

---
## Step 6 — Speech Commands v2

The dataset (~2.3 GB) downloads to `/content/datasets` (fast local disk), NOT
Drive (slow I/O bottlenecks training) and NOT the repo (gitignored). It is
re-downloaded each fresh session — that is fine, it is cached for the session.

In [ ]:
# --- 7. download + sanity-check the 12-class pipeline --------------------
# First call downloads and extracts (~2.3 GB, a few minutes). build_dataloaders
# handles the official 80:10:10 split, unknown subsampling and silence.
from data.speech_commands import build_dataloaders, class_names
from train.config import load_config
import collections

cfg = load_config('configs/base.yaml')
cfg.data.root = '/content/datasets/speech_commands_v2'

train_loader, val_loader, test_loader = build_dataloaders(
    cfg.data, batch_size=cfg.train.batch_size, sample_rate=cfg.afe.sample_rate)

print('classes:', class_names())
for name, dl in [('train', train_loader), ('val', val_loader), ('test', test_loader)]:
    print(f'{name:5s}: {len(dl.dataset):>6} clips, {len(dl)} batches')

waves, labels = next(iter(train_loader))
print('batch waveforms', tuple(waves.shape), 'labels', tuple(labels.shape))
print('label histogram', dict(sorted(collections.Counter(labels.tolist()).items())))

In [ ]:
# --- 8a. single training run (base config, C=64 T=128) ------------------
# ~85% is the target (CLAUDE.md 1). Writes runs/sc_v2/{history.json,best.pt}.
!python -m train.train --config configs/base.yaml \
    data.root=/content/datasets/speech_commands_v2 --tag sc_v2
# (data.root override is read by load_config's dotted overrides; if your shell
#  quoting fights you, edit cfg.data.root in cell 7 and call trainer directly.)

In [ ]:
# --- 8b. training from Python (equivalent to 8a, easier to tweak) --------
from data.afe import AFEFrontend
from models.binary_matchboxnet import BinaryMatchboxNet
from train.train import Trainer, set_seed

cfg = load_config('configs/base.yaml', {'tag': 'sc_v2'})
cfg.data.root = '/content/datasets/speech_commands_v2'

set_seed(cfg.train.seed)
afe = AFEFrontend(cfg.afe)
model = BinaryMatchboxNet(cfg.model)
print(model.describe())

train_loader, val_loader, test_loader = build_dataloaders(
    cfg.data, cfg.train.batch_size, cfg.afe.sample_rate, seed=cfg.train.seed)
afe.init_thresholds(next(iter(train_loader))[0])   # Cerutti IV-A

trainer = Trainer(cfg, model, afe=afe)
trainer.fit(train_loader, val_loader)
test = trainer.evaluate(test_loader)
print(f"\ntest acc {test['acc']:.4f}  ({'MEETS' if test['acc']>=0.85 else 'below'} 85%)")

In [ ]:
# --- 9. (C, T) sweep -- the milestone: smallest config clearing 85% -----
# Long. Start narrow (e.g. --C 32 64 --T 64 128) to gauge per-point time.
# Appends to experiments/results/sweep.json, which IS meant to be committed.
!python -m experiments.sweep --config configs/base.yaml \
    --C 16 32 48 64 --T 40 64 96 128 --epochs 100 \
    data.root=/content/datasets/speech_commands_v2 || \
  echo 'if the data.root override errors, set cfg.data.root in code and call run_point()'

### Saving results back to git

`experiments/results/sweep.json` is small and version-controlled. Copy it out
of Colab (download via the Files pane, or paste the summary table locally) and
commit it from your Mac — keeps history clean. Checkpoints stay in Drive; never
push `.pt` files.